kiểm tra tập dữ liệu có sử dụng được không?:


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import numpy as np
#
def evaluate_rice_dataset_1d(input_pixel_sizes):
    """
    Hàm đánh giá TỔNG THỂ TẬP DỮ LIỆU mẫu lúa thực tế.
    Phương sai được tính thủ công: Tổng (kích thước từng hạt - trung bình)^2 / số lượng hạt.
    """
    if input_pixel_sizes is None or len(input_pixel_sizes) == 0:
        return {
            "trung_binh": 0.0,
            "phuong_sai": 0.0,
            "ket_qua": False
        }

    # 1. Chuyển đổi LIST thực tế thành mảng Numpy để tính toán
    sizes_arr = np.array(input_pixel_sizes, dtype=float)

    # 2. Thuật toán tự tìm ranh giới động cho giống lúa (Lọc nhiễu bằng IQR)
    q1, q3 = np.percentile(sizes_arr, [25, 75])
    iqr = q3 - q1
    lower_limit = q1 - 1.5 * iqr
    upper_limit = q3 + 1.5 * iqr

    # 3. Trích xuất tập hạt lúa đạt chuẩn (loại bỏ lép và dính chùm)
    standard_seeds = sizes_arr[(sizes_arr >= lower_limit) & (sizes_arr <= upper_limit)]

    # --- TÍNH TOÁN GIÁ TRỊ TRUNG BÌNH THỰC TẾ ---
    if len(standard_seeds) > 0:
        optimized_mean = float(np.mean(standard_seeds))

        # --- BẮT ĐẦU TÍNH PHƯƠNG SAI THEO CÔNG THỨC THỦ CÔNG ---
        # Bước a: Lấy kích thước từng hạt trong tập sạch trừ đi số trung bình, rồi bình phương lên
        squared_errors = (standard_seeds - optimized_mean) ** 2

        # Bước b: Tính tổng các sai số bình phương đó lại và chia cho tổng số lượng hạt sạch (N)
        optimized_variance = float(np.sum(squared_errors) / len(standard_seeds))
    else:
        optimized_mean = float(np.median(sizes_arr))
        optimized_variance = 0.0

    # 4. ĐÁNH GIÁ XEM TẬP DỮ LIỆU CÓ ĐẠT CHUẨN HAY KHÔNG (Phân ngưỡng tỷ lệ % thương phẩm)
    valid_rice_count = int(np.sum(sizes_arr <= upper_limit))
    standard_seeds_count = len(standard_seeds)
    standard_rate = standard_seeds_count / valid_rice_count if valid_rice_count > 0 else 0
    final_decision = bool(standard_rate >= 0.80)

    # 5. TRẢ VỀ ĐÚNG 1 SET GỒM 3 KEY CHÍNH CỦA CẢ TẬP DỮ LIỆU
    return {
        "trung_binh": round(optimized_mean, 2),
        "phuong_sai": round(optimized_variance, 2),
        "ket_qua": final_decision
    }
    # này có loại bỏ

In [ ]:
import math
def check_standard(sizes):
    n = len(sizes)
    if n == 0:
        raise ValueError("Ds rỗng")
    mean = sum(sizes) / n
    variance = sum((x - mean) ** 2 for x in sizes) / n
    arr = np.array(sizes, dtype=float)

    q1, q3 = np.percentile(arr, [25, 75])
    iqr = q3 - q1

    lower_limit = q1 - 1.5 * iqr
    upper_limit = q3 + 1.5 * iqr

    # Đếm số hạt nằm trong khoảng chuẩn
    standard_count = np.sum(
        (arr >= lower_limit) & (arr <= upper_limit)
    )

    standard_rate = standard_count / n

    return {
        "avg": mean,
        "var": variance,
        "result": bool(standard_rate >= 0.8)
    }
  # số hạt nằm trong khoảng chuẩn mà lớn hơn 80% thì tập
  # dữ liệu đó ok



In [ ]:
#nên lấy cái này---------------------------------------------
import numpy as np

def evaluate_batch_uniformity(input_pixel_sizes, threshold_rate=0.80):
    """
    Đánh giá độ đồng đều của toàn bộ lô lúa.
    - threshold_rate: Ngưỡng chấp nhận (mặc định 80% hạt phải đạt chuẩn).
    """
    sizes_arr = np.array(input_pixel_sizes, dtype=float)
    total_objects = len(sizes_arr)

    if total_objects == 0:
        return {"trung_binh": 0.0, "phuong_sai": 0.0, "ty_le_dat": 0.0, "ket_qua": False}

    # 1. Tìm ranh giới chuẩn bằng Toán thống kê (IQR)
    q1, q3 = np.percentile(sizes_arr, [25, 75])
    iqr = q3 - q1
    lower_limit = q1 - 1.5 * iqr
    upper_limit = q3 + 1.5 * iqr

    # 2. Đếm số lượng hạt nằm trong vùng an toàn
    standard_seeds = sizes_arr[(sizes_arr >= lower_limit) & (sizes_arr <= upper_limit)]
    standard_count = len(standard_seeds)

    # 3. Tính Tỷ lệ thương phẩm (Dựa trên TOÀN BỘ đối tượng đưa vào)
    standard_rate = standard_count / total_objects

    # 4. Quyết định (True = Chấp nhận lô lúa | False = Từ chối)
    final_decision = bool(standard_rate >= threshold_rate)

    # (Tùy chọn) Tính trung bình và phương sai trên tập hạt sạch để báo cáo
    if standard_count > 0:
        mean_clean = float(np.mean(standard_seeds))
        var_clean = float(np.var(standard_seeds))
    else:
        mean_clean, var_clean = 0.0, 0.0

    return {
        "trung_binh": round(mean_clean, 2),
        "phuong_sai": round(var_clean, 2),
        "ty_le_dat": round(standard_rate * 100, 2), # Trả về % cho dễ nhìn
        "ket_qua": final_decision
    }

In [ ]:
!pip install opencv-python

In [ ]:
import os
import cv2
import numpy as np

def get_grain_areas_from_folder(folder_path):
    """
    Hàm tính diện tích các hạt lúa (đã cắt polygon) trong một thư mục.

    Input:
        - folder_path (str): Đường dẫn đến thư mục chứa các file .png.

    Output:
        - areas_list (list of floats): Danh sách các giá trị diện tích (pixel vuông).
    """
    areas_list = []
    SCALE = 4  # Hệ số phóng to nội suy để khử răng cưa viền hạt

    # Duyệt qua tất cả các file trong thư mục
    for filename in os.listdir(folder_path):
        if not filename.lower().endswith(".png"):
            continue

        path = os.path.join(folder_path, filename)

        # 1. Đọc ảnh và ép lấy lớp Alpha (trong suốt)
        img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
        if img is None or len(img.shape) < 3 or img.shape[2] < 4:
            continue

        # 2. Tạo mặt nạ từ kênh Alpha (0: trong suốt, >0: hạt lúa)
        alpha = img[:, :, 3]
        mask = np.where(alpha > 0, 255, 0).astype(np.uint8)

        # 3. Phóng to mặt nạ để bo tròn đường viền (tăng độ chính xác)
        mask = cv2.resize(mask, None, fx=SCALE, fy=SCALE, interpolation=cv2.INTER_CUBIC)

        # 4. Tìm đường viền của mặt nạ
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        if len(contours) == 0:
            continue

        # Lấy viền bao quanh mảng trắng lớn nhất (chính là hạt lúa)
        contour = max(contours, key=cv2.contourArea)

        # 5. Tính diện tích (Nhớ chia lại cho SCALE^2 do ảnh đã bị phóng to trước đó)
        area = cv2.contourArea(contour) / (SCALE ** 2)

        # Thêm vào mảng kết quả
        areas_list.append(area)

    return areas_list

In [ ]:
INPUT_DIR = "/content/drive/MyDrive/NGHIÊN CỨU KHOA HỌC/GROUP_MEMBERS/NGUYEN MINH TRI/MAIN_SOURCES/DETECTED_OBJECTS/35_special_images_segmentation.v1i.yolov8_v1_trained/CROPPED_DETECTED_OBJECTS/hat_nguyen"

danh_sach_dien_tich = get_grain_areas_from_folder(INPUT_DIR)
print(danh_sach_dien_tich)
print("Min:", min(danh_sach_dien_tich))
print("Max:", max(danh_sach_dien_tich))
print("Mean:", sum(danh_sach_dien_tich)/len(danh_sach_dien_tich))


ket_qua_danh_gia = evaluate_rice_dataset_1d(danh_sach_dien_tich)
print(ket_qua_danh_gia)

result = check_standard(danh_sach_dien_tich)
print(result)

kcs_report = evaluate_batch_uniformity(danh_sach_dien_tich, threshold_rate=0.90)
print(kcs_report)

[608.1875, 638.1875, 836.03125, 572.96875, 541.625, 588.4375, 1032.40625, 566.75, 1081.84375, 905.28125, 709.625, 553.71875, 812.53125, 682.78125, 967.90625, 579.6875, 564.59375, 494.46875, 661.78125, 498.40625, 756.59375, 695.40625, 702.09375, 765.28125, 621.78125, 435.90625, 595.84375, 633.09375, 894.5, 1087.78125, 685.53125, 598.40625, 792.96875, 996.4375, 837.875, 597.0625, 544.65625, 510.9375, 769.65625, 715.1875, 543.40625, 641.46875, 612.5, 833.59375, 581.6875, 1105.28125, 445.6875, 532.6875, 657.96875, 656.6875, 480.875, 965.84375, 549.84375, 854.125, 644.0, 7618.125, 10324.25, 7091.4375, 6268.3125, 10969.65625, 6017.625, 6474.6875, 9564.25, 9185.5, 3365.53125, 8391.625, 6978.21875, 10417.5625, 8862.4375, 8346.65625, 5200.625, 8398.40625, 9582.0, 9143.3125, 10459.03125, 8795.28125, 7882.5625, 8915.25, 7642.1875, 6728.125, 11205.5625, 6204.3125, 8707.71875, 8168.78125, 9542.96875, 4630.78125, 9301.03125, 6750.15625, 19124.90625, 23315.28125, 9092.03125, 28062.40625, 26935.90625,